# Environmental Data Extraction — NRS09 Deployment

Extracts and visualises environmental co-variates at the **NRS09** hydrophone
site in the Gulf of Maine using `ecosound.environment` and `ecosound.visualization`.

| Parameter | Value |
|---|---|
| Site | NRS09 |
| Latitude | 42.40382 °N |
| Longitude | −70.12225 °E |
| Depth | 78 m |
| Deployment period | 2018-08-01 → 2018-08-31 |

**Sections**
1. ERA5 — 10 m Wind & Precipitation
2. NECOFS — Ocean water-column profile
3. Tides — Water level & tidal index
4. Lunar & Solar ephemeris
5. Sea Surface Temperature (ERDDAP)
6. Chlorophyll-a (ERDDAP)
7. Vessel traffic (AIS)
8. Combined summary

## Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

warnings.filterwarnings('ignore')
%matplotlib inline

# --- ecosound imports ---
from ecosound.environment import (
    ERA5, NECOFS, Tides, LunarSolar,
    ERDDAPDataFetcher, AISQueryHelper,
)
from ecosound.visualization import GridPlotter, AISMapPlotter

print('All imports OK')

In [ ]:
# ============================================================
# Site and deployment period — edit here to change everything
# ============================================================

SITE_NAME  = 'NRS09'
LAT        = 42.40382    # decimal degrees N
LON        = -70.12225   # decimal degrees E (negative = West)
DEPTH_M    = 78

START_DT   = '2018-08-01'
END_DT     = '2018-08-31'

# Bounding box for gridded data (±2° lat, ±3° lon around site)
LAT_MIN = LAT - 2.0;  LAT_MAX = LAT + 2.0
LON_MIN = LON - 3.0;  LON_MAX = LON + 3.0

# Path to local AIS DuckDB database
AIS_DB = (
    r'C:\Users\xavier.mouy\Documents\GitHub'
    r'\NERACOOS_processing_scripts\non_acoustic_data'
    r'\ais_db\gulf_of_maine_ais.duckdb'
)

# Output directory for interactive HTML maps (Folium)
OUTDIR = os.path.join(os.getcwd(), 'NRS09_figures')
os.makedirs(OUTDIR, exist_ok=True)

print(f'Site  : {SITE_NAME}  ({LAT}°N, {LON}°E, {DEPTH_M} m)')
print(f'Period: {START_DT}  →  {END_DT}')
print(f'HTML maps: {OUTDIR}')

---
## 1  ERA5 — 10 m Wind & Precipitation

**Source**: Open-Meteo API (ERA5 reanalysis, free, no registration)  
**Resolution**: Hourly, 0.25° (~28 km)

In [ ]:
era5 = ERA5(source='open_meteo', verbose=True)

# --- Wind ---
ds_wind = era5.get_wind_timeseries(
    lat=LAT, lon=LON,
    start_dt=START_DT,
    end_dt=END_DT,
)
print(ds_wind)

In [ ]:
era5.plot_wind_timeseries(display=True)

In [ ]:
# --- Precipitation ---
ds_precip = era5.get_precipitation_timeseries(
    lat=LAT, lon=LON,
    start_dt=START_DT,
    end_dt=END_DT,
)
era5.plot_precipitation_timeseries(display=True)

In [ ]:
# Quick stats
times_wind = pd.to_datetime(ds_wind.time.values)
print(f"ERA5 wind — {len(times_wind)} hourly records")
print(f"  Mean wind speed : {float(ds_wind.wind_speed_ms.mean()):.2f} m/s")
print(f"  Max  wind speed : {float(ds_wind.wind_speed_ms.max()):.2f} m/s")
print(f"  Total precip    : {float(ds_precip.precipitation_mmh.sum()):.1f} mm")

---
## 2  NECOFS — Ocean Water-Column Profile

**Source**: FVCOM-GOM3 30-year hindcast via OPeNDAP  
**Coverage**: Gulf of Maine (1978 – ~2016)

> **Note**: the 30-year hindcast ends around 2016.  
> For dates after that, replace the URL with `NECOFS.GOM3_FORECAST_URL`  
> or change `necofs_date` to a date within the hindcast period.

In [ ]:
from datetime import datetime

# Use a date within the hindcast period
# Change to START_DT if using the forecast URL
NECOFS_DATE = datetime(2015, 8, 15, 12, 0, 0)

necofs = NECOFS(verbose=True)
# For operational forecast (recent dates) uncomment:
# necofs = NECOFS(url=NECOFS.GOM3_FORECAST_URL)

# --- Single profile (snapshot) ---
ds_profile = necofs.get_vertical_profile(
    lat=LAT, lon=LON,
    dt=NECOFS_DATE,
)
print(ds_profile)

In [ ]:
necofs.plot_vertical_profile(display=True)

In [ ]:
# --- Time series of profiles over one week (hindcast period) ---
ds_profiles = necofs.get_vertical_profiles(
    lat=LAT, lon=LON,
    start_dt=datetime(2015, 8, 1),
    end_dt=datetime(2015, 8, 7),
)
print(ds_profiles)

In [ ]:
necofs.plot_vertical_profiles(cmap='plasma', display=True)

In [ ]:
# --- Surface and bottom sound speed over the week ---
ss_surface = ds_profiles.sound_speed_ms.isel(sigma_layer=0)   # surface
ss_bottom  = ds_profiles.sound_speed_ms.isel(sigma_layer=-1)  # bottom
times_nc   = pd.to_datetime(ds_profiles.time.values)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times_nc, ss_surface.values, label='Surface', color='steelblue', lw=1.5)
ax.plot(times_nc, ss_bottom.values,  label='Bottom',  color='darkorange', lw=1.5)
ax.axhline(ds_profile.sound_speed_ms.mean().item(), color='gray',
           lw=0.8, ls='--', label='Profile mean')
ax.set_ylabel('Sound speed (m/s)', fontsize=10)
ax.set_title(f'NECOFS Sound Speed at {SITE_NAME} — 2015-08-01 to 2015-08-07', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, ls=':', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.show()

---
## 3  Tides — Water Level & Tidal Index

**Source**: NOAA CO-OPS API (nearest tide gauge)  
**Resolution**: 6-minute

In [ ]:
tides = Tides(verbose=True)

# Find and print the nearest gauge
sid, sname, slat, slon = tides.find_nearest_station(lat=LAT, lon=LON)
print(f'Nearest gauge: {sname} (ID {sid})  {slat:.3f}°N  {slon:.3f}°E')

In [ ]:
# Astronomical tide predictions + tidal index
ds_tide = tides.get_water_level(
    lat=LAT, lon=LON,
    start_dt=START_DT,
    end_dt=END_DT,
    product='predictions',     # clean astronomical signal
    datum='MLLW',
    compute_tidal_index=True,  # adds time_since_high_tide_h and tidal_phase
)
print(ds_tide)
print(f'\nHigh tides detected: {len(tides.high_tide_times)}')
print(f'Low  tides detected: {len(tides.low_tide_times)}')

In [ ]:
tides.plot_water_level(display=True)

In [ ]:
# Tidal phase histogram
tidal_phase = ds_tide.tidal_phase.values
tidal_phase = tidal_phase[~np.isnan(tidal_phase)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(tidal_phase, bins=24, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Tidal phase (0=high tide, ~0.5=low tide)', fontsize=9)
axes[0].set_ylabel('Count (6-min intervals)', fontsize=9)
axes[0].set_title('Tidal Phase Distribution', fontsize=10)
axes[0].grid(True, ls=':', alpha=0.5)

axes[1].hist(ds_tide.time_since_high_tide_h.values[
    ~np.isnan(ds_tide.time_since_high_tide_h.values)],
    bins=24, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Hours since last high tide', fontsize=9)
axes[1].set_ylabel('Count (6-min intervals)', fontsize=9)
axes[1].set_title('Time Since High Tide Distribution', fontsize=10)
axes[1].grid(True, ls=':', alpha=0.5)

fig.suptitle(f'Tidal Statistics at {SITE_NAME} — {START_DT} to {END_DT}', fontsize=11)
plt.tight_layout()
plt.show()

---
## 4  Lunar & Solar Ephemeris

**Source**: PyEphem (computed locally, no internet required)  
**Resolution**: Hourly (configurable)

In [ ]:
ls = LunarSolar(verbose=True)

ds_ephem = ls.get_timeseries(
    lat=LAT, lon=LON,
    start_dt=START_DT,
    end_dt=END_DT,
    freq='1h',
)
print(ds_ephem)

In [ ]:
ls.plot_timeseries(display=True)

In [ ]:
# Day / night summary
n_total  = len(ds_ephem.time)
n_day    = int(ds_ephem.is_day.sum())
n_ctwi   = int(ds_ephem.is_civil_twilight.sum())
n_ntwi   = int(ds_ephem.is_nautical_twilight.sum())
n_night  = int(ds_ephem.is_night.sum())

labels  = ['Day', 'Civil\ntwilight', 'Nautical\ntwilight', 'Night']
counts  = [n_day, n_ctwi, n_ntwi, n_night]
colors  = ['lightyellow', 'peachpuff', 'lightsalmon', 'midnightblue']

fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    counts,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'gray', 'linewidth': 0.8},
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title(f'Light conditions at {SITE_NAME} — {START_DT} to {END_DT}\n(hourly)', fontsize=10)
plt.tight_layout()
plt.show()

print(f'Lunar illumination — mean: {float(ds_ephem.moon_illumination_pct.mean()):.1f}%  '
      f'max: {float(ds_ephem.moon_illumination_pct.max()):.1f}%')

---
## 5  Sea Surface Temperature (ERDDAP)

**Source**: NOAA ACSPO SST reanalysis via NEFSC ERDDAP  
**Dataset**: `noaa_coastwatch_acspo_v2_reanalysis`  
**Resolution**: Daily, ~2 km

In [ ]:
fetcher_sst = ERDDAPDataFetcher(
    server='https://comet.nefsc.noaa.gov/erddap',
    dataset_id='noaa_coastwatch_acspo_v2_reanalysis',
)

# Fetch SST over the Gulf of Maine bounding box (best-quality only)
ds_sst = fetcher_sst.fetch_data(
    'sea_surface_temperature',
    start_date=START_DT,
    end_date=END_DT,
    lat_min=LAT_MIN, lat_max=LAT_MAX,
    lon_min=LON_MIN, lon_max=LON_MAX,
    quality_mask_value=5,         # keep only best-quality pixels
    max_request_duration_days=31,
)
print(ds_sst)

In [ ]:
# --- Time series at NRS09 (nearest grid cell) ---
sst_nrs09 = ds_sst.sea_surface_temperature.sel(
    latitude=LAT, longitude=LON, method='nearest'
)
times_sst = pd.to_datetime(sst_nrs09.time.values)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times_sst, sst_nrs09.values, color='crimson', lw=1.5, marker='o', ms=3)
ax.set_ylabel('Sea Surface Temperature (°C)', fontsize=10)
ax.set_title(f'SST at {SITE_NAME} — {START_DT} to {END_DT}', fontsize=11)
ax.grid(True, ls=':', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax.xaxis.get_ticklabels(), rotation=30, ha='right', fontsize=8)
print(f'SST — mean: {float(sst_nrs09.mean(skipna=True)):.2f} °C  '
      f'min: {float(sst_nrs09.min(skipna=True)):.2f} °C  '
      f'max: {float(sst_nrs09.max(skipna=True)):.2f} °C')
plt.tight_layout()
plt.show()

In [ ]:
# --- Static map for a single date (weekly mean shown) ---
recorder_data = pd.DataFrame({
    'name':      ['SB03',    'NRS09'],
    'latitude':  [42.2554,   42.40382],
    'longitude': [-70.1786, -70.12225],
    'depth_m':   [45,        78],
})

# Weekly mean
ds_sst_weekly = ds_sst.resample(time='1W').mean(skipna=True)

# Plot first available week
first_week = str(ds_sst_weekly.time.values[0])[:10]
plotter_sst = GridPlotter()
fig_sst = plotter_sst.plot_static_map(
    ds_sst_weekly.sea_surface_temperature,
    timestamp=first_week,
    label='Weekly Mean SST (°C)',
    colormap='RdYlBu_r',
    vmin=10, vmax=25,
    cbar_min=10, cbar_max=25,
    axes_fontsize=8,
    marker_type='o',
    marker_size=2,
    colorbar_fontsize=10,
    title_fontsize=11,
    recorder_df=recorder_data,
    show_recorder_names=True,
    recorder_name_fontsize=8,
    bathymetry_contours=[-200],
    bathymetry_color='black',
    bathymetry_linewidth=0.4,
    bathymetry_linestyle='-',
    bathymetry_fontsize=6,
    dpi=200,
    show=True,
)

In [ ]:
# --- Interactive HTML time-slider map (all weekly SST frames) ---
plotter_sst_html = GridPlotter(basemap='Esri Ocean', zoom_start=7)
plotter_sst_html.plot_timeseries_grid(
    ds_sst_weekly.sea_surface_temperature,
    label='Weekly Mean SST (°C)',
    colormap='RdYlBu_r',
    opacity=0.7,
    playback_speed_ms=800,
)
plotter_sst_html.add_recorder_locations(recorder_data)
plotter_sst_html.save(os.path.join(OUTDIR, 'sst_timeseries_map.html'))
print('Interactive SST map saved — open sst_timeseries_map.html in a browser')

# Display inline in Jupyter
plotter_sst_html.map

---
## 6  Chlorophyll-a (ERDDAP)

**Source**: OC-CCI v6 (ocean colour) via NEFSC ERDDAP  
**Dataset**: `occci_v6_daily_1km`  
**Resolution**: Daily, 1 km

In [ ]:
fetcher_chla = ERDDAPDataFetcher(
    server='https://comet.nefsc.noaa.gov/erddap',
    dataset_id='occci_v6_daily_1km',
)

ds_chla = fetcher_chla.fetch_data(
    'chlor_a',
    start_date=START_DT,
    end_date=END_DT,
    lat_min=LAT_MIN, lat_max=LAT_MAX,
    lon_min=LON_MIN, lon_max=LON_MAX,
    max_request_duration_days=31,
)
print(ds_chla)

In [ ]:
# --- Time series at NRS09 ---
chla_nrs09 = ds_chla.chlor_a.sel(
    latitude=LAT, longitude=LON, method='nearest'
)
times_chla = pd.to_datetime(chla_nrs09.time.values)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(times_chla, chla_nrs09.values, alpha=0.4, color='forestgreen')
ax.plot(times_chla, chla_nrs09.values, color='forestgreen', lw=1.5, marker='o', ms=3)
ax.set_ylabel('Chlorophyll-a (mg m⁻³)', fontsize=10)
ax.set_title(f'Chlorophyll-a at {SITE_NAME} — {START_DT} to {END_DT}', fontsize=11)
ax.grid(True, ls=':', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax.xaxis.get_ticklabels(), rotation=30, ha='right', fontsize=8)
print(f'Chla — mean: {float(chla_nrs09.mean(skipna=True)):.3f} mg/m³  '
      f'max: {float(chla_nrs09.max(skipna=True)):.3f} mg/m³')
plt.tight_layout()
plt.show()

In [ ]:
# --- Static map (first week) ---
ds_chla_weekly = ds_chla.resample(time='1W').mean(skipna=True)
first_week_chla = str(ds_chla_weekly.time.values[0])[:10]

plotter_chla = GridPlotter()
fig_chla = plotter_chla.plot_static_map(
    ds_chla_weekly.chlor_a,
    timestamp=first_week_chla,
    label='Weekly Mean Chlorophyll-a (mg m⁻³)',
    colormap='YlGn',
    vmin=0, vmax=5,
    cbar_min=0, cbar_max=5,
    axes_fontsize=8,
    marker_type='o',
    marker_size=2,
    colorbar_fontsize=10,
    title_fontsize=11,
    recorder_df=recorder_data,
    show_recorder_names=True,
    recorder_name_fontsize=8,
    bathymetry_contours=[-200],
    bathymetry_color='black',
    bathymetry_linewidth=0.4,
    bathymetry_linestyle='-',
    bathymetry_fontsize=6,
    dpi=200,
    show=True,
)

---
## 7  Vessel Traffic (AIS)

**Source**: Local DuckDB + Parquet database (Marine Cadastre / NOAA)  
**DB**: `gulf_of_maine_ais.duckdb`

> If the AIS database is not available, skip this section.

In [ ]:
# Check DB availability
if not os.path.exists(AIS_DB):
    print(f'AIS database not found at:\n  {AIS_DB}\nSkipping AIS section.')
    AIS_AVAILABLE = False
else:
    AIS_AVAILABLE = True
    print(f'AIS database found: {AIS_DB}')

In [ ]:
if AIS_AVAILABLE:
    # --- Statistics and unique vessels in 50 km radius ---
    with AISQueryHelper(AIS_DB) as ais:
        stats = ais.get_statistics(
            start_date=START_DT,
            end_date=END_DT,
            min_lat=LAT - 0.45,
            max_lat=LAT + 0.45,
            min_lon=LON - 0.45,
            max_lon=LON + 0.45,
        )
    print('AIS Statistics (±0.45° bounding box around NRS09):')
    for k, v in stats.items():
        print(f'  {k}: {v}')

In [ ]:
if AIS_AVAILABLE:
    # --- Query raw AIS points within 50 km of NRS09 ---
    with AISQueryHelper(AIS_DB) as ais:
        gdf_ais = ais.query_radius(
            start_date=START_DT,
            end_date=END_DT,
            center_lat=LAT,
            center_lon=LON,
            radius_km=50,
        )
    print(f'Found {len(gdf_ais)} AIS records within 50 km of {SITE_NAME}')
    print(gdf_ais[['mmsi', 'vessel_name', 'vessel_category', 'sog', 'distance_km']].head(10))

In [ ]:
if AIS_AVAILABLE:
    # --- Vessel category summary ---
    cat_counts = (
        gdf_ais.groupby('vessel_category')
               .size()
               .sort_values(ascending=False)
               .reset_index(name='n_records')
    )
    print(cat_counts.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(cat_counts.vessel_category, cat_counts.n_records,
            color='steelblue', edgecolor='white')
    ax.set_xlabel('Number of AIS records', fontsize=10)
    ax.set_title(f'Vessel categories within 50 km of {SITE_NAME}\n{START_DT} to {END_DT}',
                 fontsize=11)
    ax.grid(True, axis='x', ls=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
if AIS_AVAILABLE:
    # --- Gridded vessel counts (1 km × 1 km, hourly) ---
    with AISQueryHelper(AIS_DB) as ais:
        vessel_grid = ais.create_gridded_vessel_counts(
            start_date=START_DT,
            end_date=END_DT,
            min_lat=LAT - 0.45,
            max_lat=LAT + 0.45,
            min_lon=LON - 0.45,
            max_lon=LON + 0.45,
            width_km=1,
            height_km=1,
            time_resolution_hours=1,
        )
    print(vessel_grid)

In [ ]:
if AIS_AVAILABLE and vessel_grid is not None:
    # Daily vessel-count time series (sum over grid cells)
    daily_count = (
        vessel_grid
        .sum(dim=['latitude', 'longitude'], skipna=True)
        .resample(time='1D').sum(skipna=True)
    )
    times_ais = pd.to_datetime(daily_count.time.values)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(times_ais, daily_count.values, color='steelblue',
           width=0.8, edgecolor='white', alpha=0.85)
    ax.set_ylabel('Daily AIS record count\n(1 km grid, 50 km radius)', fontsize=9)
    ax.set_title(f'Daily Vessel Presence near {SITE_NAME} — {START_DT} to {END_DT}', fontsize=11)
    ax.grid(True, ls=':', alpha=0.5, axis='y')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.setp(ax.xaxis.get_ticklabels(), rotation=30, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
if AIS_AVAILABLE and vessel_grid is not None:
    # --- Static map: cumulative vessel count over the month ---
    vessel_grid_sum = vessel_grid.sum(dim='time', skipna=True, min_count=1)

    plotter_ais = GridPlotter()
    fig_ais = plotter_ais.plot_static_map(
        vessel_grid_sum,
        timestamp=START_DT,
        label=f'Monthly Cumulative Vessel Count ({START_DT[:7]})',
        colormap='YlOrRd',
        vmin=0, vmax=500,
        cbar_min=0, cbar_max=500,
        axes_fontsize=8,
        marker_type='o',
        marker_size=2,
        colorbar_fontsize=10,
        title_fontsize=11,
        recorder_df=recorder_data,
        show_recorder_names=True,
        recorder_name_fontsize=9,
        dpi=200,
        show=True,
    )

In [ ]:
if AIS_AVAILABLE:
    # --- Interactive map: raw AIS positions coloured by vessel category ---
    ais_plotter = AISMapPlotter(
        basemap='Esri Ocean',
        zoom_start=10,
        db_path=AIS_DB,    # used to build colour scheme from DB lookup table
    )
    ais_plotter.plot_ais_data(
        gdf_ais,
        color_by='vessel_category',
        color_map=ais_plotter.COLOR_SCHEMES['vessel_category'],
        popup_fields=['mmsi', 'vessel_name', 'vessel_category',
                      'sog', 'cog', 'distance_km', 'base_datetime'],
        marker_size=5,
    )
    ais_plotter.add_recorder_locations(recorder_data)
    ais_plotter.save(os.path.join(OUTDIR, 'ais_map_by_category.html'))
    print('Interactive AIS map saved — open ais_map_by_category.html in a browser')

    # Display inline in Jupyter
    ais_plotter.map

---
## 8  Combined Summary

Merge all hourly time-series datasets onto a common time axis and produce
a single multi-panel figure covering the full deployment period.

In [ ]:
# Common hourly time axis (from ERA5 wind, already hourly)
time_ref = ds_wind.time.values

# Resample / reindex each dataset to the ERA5 hourly time axis
wind_speed = ds_wind.wind_speed_ms.values
wind_dir   = ds_wind.wind_dir_deg.values

# Tides → nearest 6-min record to each hour
wl_1h = (
    ds_tide.water_level_m
    .reindex(time=time_ref, method='nearest', tolerance='10min')
    .values
)
phase_1h = (
    ds_tide.tidal_phase
    .reindex(time=time_ref, method='nearest', tolerance='10min')
    .values
)

# Lunar/Solar → already hourly (same grid if freq='1h' and same start/end)
moon_ill = (
    ds_ephem.moon_illumination_pct
    .reindex(time=time_ref, method='nearest', tolerance='1h')
    .values
)
sun_alt = (
    ds_ephem.sun_altitude_deg
    .reindex(time=time_ref, method='nearest', tolerance='1h')
    .values
)

# SST → daily, reindex to hourly (fill with nearest daily value)
sst_1h = (
    sst_nrs09
    .reindex(time=time_ref, method='nearest', tolerance='24h')
    .values
)

# Chla → daily
chla_1h = (
    chla_nrs09
    .reindex(time=time_ref, method='nearest', tolerance='24h')
    .values
)

times_plot = pd.to_datetime(time_ref)
print(f'Common time axis: {len(times_plot)} hourly steps from {times_plot[0].date()} to {times_plot[-1].date()}')

In [ ]:
# Night-time shading helper
def shade_night(ax, times, sun_alt, color='lightsteelblue', alpha=0.20):
    """Shade periods where the sun is below the horizon."""
    is_dark = sun_alt <= 0
    ax.fill_between(times, ax.get_ylim()[0], ax.get_ylim()[1],
                    where=is_dark, color=color, alpha=alpha, zorder=0)

n_panels = 5 if not AIS_AVAILABLE else 6
fig, axes = plt.subplots(n_panels, 1,
                          figsize=(14, 3.2 * n_panels),
                          sharex=True)

kw_grid  = dict(ls=':', lw=0.5, alpha=0.6)
kw_night = dict(color='midnightblue', alpha=0.12)

ax_idx = 0

# --- Panel 1: Wind speed ---
ax = axes[ax_idx]; ax_idx += 1
ax.plot(times_plot, wind_speed, color='steelblue', lw=0.9, label='Wind speed')
ax.fill_between(times_plot, wind_speed, alpha=0.25, color='steelblue')
ax.set_ylabel('Wind speed (m/s)', fontsize=9)
ax.set_ylim(0, max(wind_speed) * 1.15)
ax.fill_between(times_plot, 0, max(wind_speed) * 1.15, where=sun_alt <= 0,
                color='midnightblue', alpha=0.10, zorder=0)
ax.grid(**kw_grid)
ax.tick_params(labelsize=8)
ax.set_title(f'ERA5 10 m wind speed', fontsize=9, loc='left')

# --- Panel 2: Water level ---
ax = axes[ax_idx]; ax_idx += 1
ax.plot(times_plot, wl_1h, color='royalblue', lw=0.7, label='Water level')
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.set_ylabel('Water level (m, MLLW)', fontsize=9)
ylo, yhi = np.nanmin(wl_1h) - 0.2, np.nanmax(wl_1h) + 0.2
ax.set_ylim(ylo, yhi)
ax.fill_between(times_plot, ylo, yhi, where=sun_alt <= 0,
                color='midnightblue', alpha=0.10, zorder=0)
ax.grid(**kw_grid)
ax.tick_params(labelsize=8)
ax.set_title('NOAA CO-OPS water level (predictions)', fontsize=9, loc='left')

# --- Panel 3: SST ---
ax = axes[ax_idx]; ax_idx += 1
ax.plot(times_plot, sst_1h, color='crimson', lw=1.2, label='SST', zorder=3)
ax.set_ylabel('SST (°C)', fontsize=9)
ylo, yhi = np.nanmin(sst_1h) - 0.5, np.nanmax(sst_1h) + 0.5
ax.set_ylim(ylo, yhi)
ax.fill_between(times_plot, ylo, yhi, where=sun_alt <= 0,
                color='midnightblue', alpha=0.10, zorder=0)
ax.grid(**kw_grid)
ax.tick_params(labelsize=8)
ax.set_title('Sea Surface Temperature (NOAA ACSPO, nearest grid cell)', fontsize=9, loc='left')

# --- Panel 4: Chlorophyll-a ---
ax = axes[ax_idx]; ax_idx += 1
ax.fill_between(times_plot, chla_1h, alpha=0.35, color='forestgreen', zorder=2)
ax.plot(times_plot, chla_1h, color='forestgreen', lw=1.0, zorder=3)
ax.set_ylabel('Chlorophyll-a (mg m⁻³)', fontsize=9)
ax.set_ylim(0, np.nanmax(chla_1h) * 1.2 if np.nanmax(chla_1h) > 0 else 1)
ax.grid(**kw_grid)
ax.tick_params(labelsize=8)
ax.set_title('Chlorophyll-a (OC-CCI v6, nearest grid cell)', fontsize=9, loc='left')

# --- Panel 5: Lunar illumination ---
ax = axes[ax_idx]; ax_idx += 1
ax.fill_between(times_plot, moon_ill, alpha=0.4, color='mediumpurple', zorder=2)
ax.plot(times_plot, moon_ill, color='mediumpurple', lw=0.9, zorder=3)
ax.set_ylabel('Lunar illumination (%)', fontsize=9)
ax.set_ylim(-2, 112)
ax.grid(**kw_grid)
ax.tick_params(labelsize=8)
ax.set_title('Lunar illumination — shaded = night (sun alt ≤ 0°)', fontsize=9, loc='left')
ax.fill_between(times_plot, -2, 112, where=sun_alt <= 0,
                color='midnightblue', alpha=0.12, zorder=0)

# --- Panel 6 (optional): AIS daily vessel count ---
if AIS_AVAILABLE and vessel_grid is not None:
    ax = axes[ax_idx]; ax_idx += 1
    daily_sum = (
        vessel_grid
        .sum(dim=['latitude', 'longitude'], skipna=True)
        .resample(time='1D').sum(skipna=True)
    )
    t_daily = pd.to_datetime(daily_sum.time.values)
    ax.bar(t_daily, daily_sum.values, color='darkorange', edgecolor='white',
           alpha=0.85, width=0.8, label='AIS records')
    ax.set_ylabel('Daily AIS records\n(1 km grid)', fontsize=9)
    ax.grid(**kw_grid, axis='y')
    ax.tick_params(labelsize=8)
    ax.set_title('Vessel presence (50 km radius, AIS)', fontsize=9, loc='left')

# --- Shared x-axis formatting ---
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(axes[-1].xaxis.get_ticklabels(), rotation=30, ha='right', fontsize=8)
axes[-1].set_xlabel('Date (UTC)', fontsize=9)

fig.suptitle(
    f'Environmental Summary — {SITE_NAME} ({LAT:.4f}°N, {LON:.4f}°E, {DEPTH_M} m)\n'
    f'{START_DT}  →  {END_DT}\n'
    f'(blue shading = night)',
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

## Interactive HTML maps

Two Folium maps are saved to `NRS09_figures/` and can be opened in any browser:

| File | Content |
|---|---|
| `sst_timeseries_map.html` | SST weekly time-slider map (Folium) |
| `ais_map_by_category.html` | Raw AIS positions coloured by vessel category (Folium) |

All other figures are rendered inline above.